# `01_dataset_exploration.ipynb` — GermEval 2017 Corpus Audit & Task Structure (0–15 min CAS Session)

## 1. Domain & Shared Task Background
The **GermEval 2017 Shared Task on Aspect-based Sentiment in Social Media Customer Feedback** (Wojatzki et al., 2017) consists of **26,209 German social media posts, tweets, forum comments, and news snippets** discussing **Deutsche Bahn (DB)** and public rail transport.

### Why Four Splits? Synchronic vs. Diachronic Evaluation
- **`train_v1.4` (19,432 docs)** and **`dev_v1.4` (2,369 docs):** Collected between May 2015 and June 2016.
- **`test_TIMESTAMP1` (`test_syn`, 2,566 docs — Synchronic Test):** Drawn from the **same time period** as train/dev. Measures in-distribution generalization.
- **`test_TIMESTAMP2` (`test_dia`, 1,842 docs — Diachronic Test):** Collected months later (**November 2016 – January 2017**). Features new winter weather disruptions, new tariff discussions, and shifting social media topics. Measures **temporal robustness**.

### Five Critical Corpus Properties Every Practitioner Must Know
1. **Irrelevant Documents (`relevance=false`, ~16–18% of corpus):** Posts that mention railway keywords (e.g., *"Bahnhof"*, *"Zug"*) in unrelated contexts (such as soccer transfers *"Zug zum Tor"* or Mallorca real estate) have **zero opinions** (`{"labels": []}`). Models that blindly predict a label on every input suffer heavy false-positive penalties under strict evaluation.
2. **Severe Long-Tail Category Imbalance:** The top 3 categories (`Allgemein`, `Zugfahrt`, `Sonstige_Unregelmässigkeiten`) account for **66% of all annotations**, while rare categories like `Gepäck` (46 in train), `Reisen_mit_Kindern` (104), and `Barrierefreiheit` (97) represent `<0.5%` each.
3. **Implicit (`NULL`) vs. Explicit Targets:** Roughly **44% of all opinions have `target="NULL"` (`from=0, to=0`)**—meaning the user expresses an aspect opinion without a localized noun/verb target span. These count in **Task C** (document-level classification) but are excluded from **Task D** (target extraction).
4. **German Discontinuous Multi-Word Targets (`' * '`):** In German syntax, separable prefix verbs (*"fällt ... aus"*, *"zeigt ... an"*) and circumpositions split a single semantic target across non-adjacent tokens. Annotators joined the pieces with `' * '` (`938 / 12,139` explicit targets in train, `7.7%`).
5. **Gold XML Typos Discovered in Our Audit:**
   - `polarity="positve"` (missing `i`) in `train_v1.4`
   - `polarity=" negative"` (leading space) in `test_TIMESTAMP2`
   - `category="Sichaupt"` (corrupted `Sicherheit#Haupt`) in `train_v1.4`
   - `category="QR-Code#Haupt"` (unseen 20th category appearing once in `dev` and once in `test_syn`, 0 times in `train`)

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
stats = json.loads(Path('data/manifests/statistics.json').read_text())
rows = []
for sp, d in stats['splits'].items():
    rows.append({
        'split': sp,
        'docs': d['num_documents'],
        'rel_false': d['num_relevance_false'],
        'zero_op_docs': d['num_docs_zero_opinions'],
        'total_opinions': d['num_opinions_total'],
        'implicit_null': d['num_opinions_implicit'],
        'explicit_total': d['num_opinions_explicit'],
        'explicit_contiguous': d['num_explicit_exact_contiguous'],
        'explicit_star': d['num_explicit_discontinuous_star'],
        'explicit_other_mismatch': d['num_explicit_non_star_mismatch'],
        'repeated_target_in_doc': d['num_explicit_repeated_target_in_doc'],
    })
df_splits = pd.DataFrame(rows)
display(df_splits)

In [ ]:
# Display category distribution and polarity distribution plots
display(Image(filename='results/figures/dataset_category_distribution.png'))
display(Image(filename='results/figures/dataset_polarity_by_split.png'))

In [ ]:
audit = json.loads(Path('data/manifests/validation_audit.json').read_text())
print('Total validation findings (including DISCONTINUOUS_STAR_TARGET info logs):', audit['total_issues'])
non_info = [i for i in audit['issues'] if i['severity'] != 'INFO']
print('Warnings/Errors count:', len(non_info))
for item in non_info[:10]:
    print(f"[{item['split']}] {item['issue_code']}: {item['message']}")

### Hands-on Classroom Exercise
Inspect real German examples of (a) an implicit `NULL` opinion and (b) a discontinuous `' * '` opinion from `dev.jsonl`.

In [ ]:
from germeval_llm.xml_parser import load_canonical_jsonl
dev_docs = load_canonical_jsonl('data/canonical/dev.jsonl')
implicit_ex = next(d for d in dev_docs if any(o.is_implicit for o in d.opinions))
star_ex = next(d for d in dev_docs if any(o.is_discontinuous_star for o in d.opinions))

print('1. Implicit Opinion Example:', implicit_ex.text)
print('   Opinions:', [(o.category_full, o.polarity, o.target_xml) for o in implicit_ex.opinions])

print('\n2. Discontinuous (*) Target Example:', star_ex.text)
print('   Opinions:', [(o.category_full, o.polarity, o.target_xml, o.start_xml, o.end_xml) for o in star_ex.opinions if o.is_discontinuous_star])

### Key Takeaways
- Always audit raw annotation files before training an LLM: fixing `positve` and `Sichaupt` on `train_v1.4` prevents polluting the training vocabulary while leaving `dev`/`test` untouched for official comparability.
- Nearly all non-contiguous explicit targets (`938/945` in train) are German separable verbs or multi-word expressions joined with `' * '`.
- Roughly 16–18% of documents in every split are irrelevant (`relevance=false`) and have zero opinions (`{"labels": []}`).